# MedClaim — QLoRA fine-tuning on a free Colab T4 (roadmap step 1)

Runtime → Change runtime type → **T4 GPU** before running.

Flow: clone repo → install deps → HF login (gated Llama 3.2 access required) →
train LoRA adapter → sanity-test → merge → GGUF q4_K_M → push to your HF Hub repo.


In [ ]:
!nvidia-smi


## 1. Clone the (private) repo
Create a GitHub fine-grained token with read access to the MediClaim repo.


In [ ]:
from getpass import getpass
GH_USER = 'Sudhanshu-Patil'   #@param {type:'string'}
REPO = 'MediClaim'            #@param {type:'string'}
gh_token = getpass('GitHub token (repo read): ')
!git clone https://{gh_token}@github.com/{GH_USER}/{REPO}.git
%cd {REPO}


## 2. Install training dependencies


In [ ]:
!pip install -q -r finetuning/requirements-colab.txt


## 3. Hugging Face login
Needs: (a) accepted access to `meta-llama/Llama-3.2-3B-Instruct` on your HF
account, (b) a **write** token so the adapter/GGUF can be uploaded.


In [ ]:
from huggingface_hub import login
login()


## 4. Train the QLoRA adapter (~30–60 min for 2 epochs on T4)
The dataset (`finetuning/data/*.jsonl`) is committed in the repo — built
locally from the ingested policy chunks + MedQuad.


In [ ]:
!python finetuning/qlora_train.py \n    --base-model meta-llama/Llama-3.2-3B-Instruct \n    --train-file finetuning/data/train.jsonl \n    --val-file finetuning/data/val.jsonl \n    --output-dir outputs/medclaim-lora \n    --epochs 2


## 5. Sanity-test the adapter
A citation-format probe: the answer should be JSON with the chunk_id cited.


In [ ]:
import json, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE = 'meta-llama/Llama-3.2-3B-Instruct'
tok = AutoTokenizer.from_pretrained('outputs/medclaim-lora')
model = AutoModelForCausalLM.from_pretrained(
    BASE, device_map='auto',
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                           bnb_4bit_compute_dtype=torch.float16))
model = PeftModel.from_pretrained(model, 'outputs/medclaim-lora')

system = json.loads(open('finetuning/data/train.jsonl').readline())['messages'][0]['content']
user = ('CONTEXT:
[chunk_id=demo-123]
| Code | Description | Max | Copay | Auth |
'
        '|---|---|---|---|---|
| OP-3003 | MRI brain | 680.00 | 20% | Yes |

'
        'QUESTION: What is the copay for an MRI of the brain?')
prompt = tok.apply_chat_template([{'role':'system','content':system},{'role':'user','content':user}],
                                 tokenize=False, add_generation_prompt=True)
inputs = tok(prompt, return_tensors='pt').to(model.device)
out = model.generate(**inputs, max_new_tokens=120, do_sample=False)
print(tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))


## 6. Merge → GGUF (q4_K_M) → upload to your HF Hub repo
Free up the training model from VRAM first, then run the converter.
Set `HF_REPO` to a repo under your account (created automatically).


In [ ]:
del model
import gc, torch
gc.collect(); torch.cuda.empty_cache()


In [ ]:
HF_REPO = 'YOUR_HF_USERNAME/medclaim-llama3.2-3b-gguf'  #@param {type:'string'}
!python finetuning/merge_and_convert.py \n    --base-model meta-llama/Llama-3.2-3B-Instruct \n    --adapter outputs/medclaim-lora \n    --hf-repo {HF_REPO}


## 7. Deploy on your laptop
```bash
huggingface-cli download <HF_REPO> medclaim-llama3.2-3b-q4_K_M.gguf --local-dir finetuning/models
ollama create medclaim-llm -f finetuning/Modelfile
ollama run medclaim-llm
```
